# Study 821 — Turnover Volatility — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the dollar-volume variant, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 4084, 'rows': 4147, 'fingerprint': '357fd262912f', 'spread_bps': -1.7, 't_nw': -1.73, 't_1s': -1.67, 'lo_bps': 5.95, 'hi_bps': 7.65, 'welch_t': -0.65, 'gross_sharpe': -0.41, 'placebo_obs': -1.7, 'placebo_mean': 0.04, 'placebo_sd': 0.931, 'placebo_p': 0.971, 'placebo_sigma_left': 1.87, 'placebo_draws': 1000, 'era_early_bps': -3.0, 'era_early_t': -2.21, 'era_early_n': 1950, 'era_late_bps': -0.51, 'era_late_t': -0.36, 'era_late_n': 2134, 'dollar_bps': -1.64, 'dollar_t': -1.68, 'timer_1_gross': -1.7, 'timer_1_cost': 2.14, 'timer_1_net': -3.84, 'timer_1_t': -3.76, 'timer_5_gross': -1.7, 'timer_5_cost': 10.14, 'timer_5_net': -11.84, 'timer_5_t': -11.6, 'null_mean_t': 0.21, 'null_sd_t': 1.01, 'null_fire': 0, 'planted_t': 9.33, 'planted_welch': 9.65}

## The headline — long-low-vol / short-high-vol spread

Daily equal-weight bottom-30% minus top-30% turnover-CV spread.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : low-vol {R['lo_bps']:+.2f} vs high-vol {R['hi_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : -1.70 bps/day  NW(10) t = -1.73  one-sample t = -1.67
books         : low-vol +5.95 vs high-vol +7.65 bps (Welch t = -0.65)
gross Sharpe  : -0.41 (before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> right-tail p = {R['placebo_p']:.5f} "
      f"(~{R['placebo_sigma_left']:.2f} sigma into the left tail)")

observed -1.70 bps vs placebo mean +0.040 (sd 0.931) -> right-tail p = 0.97100 (~1.87 sigma into the left tail)


## Robustness — two eras (split 2018-01-01) + dollar-volume variant

In [4]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")
print(f"dollar volume (Volume x Close): {R['dollar_bps']:+.2f} bps  NW t = {R['dollar_t']:+.2f}")

2010-2017 (n=1950): -3.00 bps  NW t = -2.21
2018-2026 (n=2134): -0.51 bps  NW t = -0.36
dollar volume (Volume x Close): -1.64 bps  NW t = -1.68


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross -1.70 -> net -3.84 bps/day (cost 2.14/day, t=-3.76)
5 bps one-way: gross -1.70 -> net -11.84 bps/day (cost 10.14/day, t=-11.60)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from turnover_vol import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=821+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0016, seed=821, n_assets=40, n_days=1500))
print(f"planted (edge=0.0016): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean +0.01 (sd 1.11), |t|>=2 in 0/8


planted (edge=0.0016): NW t = +9.33, Welch t = +9.65


## Verdict

- **Signal — None.** The claimed Chordia-Subrahmanyam-Anshuman turnover-variability premium does **not** replicate on 50 liquid US mega-caps: the long-low-vol / short-high-vol spread is an insignificant **-1.70 bps/day** (NW *t* = **-1.73**, |t| < 2), faintly *wrong-signed*, carried entirely by the pre-2018 era (*t* = -2.21) and gone thereafter (*t* = -0.36); the dollar-volume variant agrees (-1.64 bps, *t* = -1.68) and the placebo shows no reliable spread. The 20-seed synthetic control recovers a *planted* relation cleanly (*t* = +9.33, fires on 0/20 nulls), so the machinery is sound — the effect is simply absent on mega-caps (a small/illiquid phenomenon). Survivorship biases the magnitude.
- **Tradability — Mirage.** The specified book loses money gross and net (-3.84 bps/day at 1 bp, *t* = -3.76; -11.84 at 5 bps); even a sign-flip is eaten by the 2.14 bps/day round-trip friction at 1 bp.